# SmogSense: Physics-Residual Meta-Quantile Learning (M5, M6 & M7)
### End-to-End GPU Training & Evaluation in Google Colab (Free Tier T4 GPU)

**Architectural Foundation:** [ADR-024: Physics-Residual Meta-Quantile Learning](https://github.com/haiderkashan/smog-sense/blob/main/decisions.md)

This notebook demonstrates the end-to-end model training, pre-training, and adaptation workflow for **SmogSense Phase 2**:
1. **LightGBM Physics Base (M5)**: 57 gradient boosted quantile regressors ($3\text{ horizons} \times 19\text{ quantiles}$) trained on Delhi source physics features with season-blocked cross-fitting on Block E (6 seasons, 3 folds) to supply un-leaked out-of-fold (OOF) base predictions.
2. **Supervised Residual Pre-Training (M6)**: Supervised pre-training of the `ResidualCalibrator` (47,577 parameters: `LocalGRUEncoder` + `ResidualQuantileHead`) on Block E out-of-fold (OOF) base predictions with synthetic low-cost sensor corruption augmentation, evaluated on Block V.
3. **Meta-SGD GRU Adapter (M7)**: A PyTorch gated recurrent unit (GRU) with `torch.func.vmap` vectorized inner-loop adaptation that predicts the local residual $\Delta \hat{z}_{i, T+h}$ on sparse target station observations.
4. **Zero-Lookahead Evaluation**: Strict point-in-time correctness, evaluation on held-out Lahore regulatory monitor (US Consulate 8664) across sensor-sparsity levels $N \in \{0, 1, 2\}$, and comparison against naive persistence (M0) and raw CAMS (M1).

In [ ]:
# 1. Environment Setup & Data Bundle Retrieval
import sys
import tarfile
import urllib.request
from pathlib import Path

import torch

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("==> Running inside Google Colab environment.")
    !git clone --depth 1 https://github.com/haiderkashan/smog-sense.git /content/smogsense_repo
    %cd /content/smogsense_repo
    !pip install -q "numpy>=2.0" "pandas>=2.2" "scipy>=1.14" "scikit-learn>=1.5" "lightgbm>=4.3" "torch>=2.2" "pyarrow>=16.0" "matplotlib>=3.8"
    if "/content/smogsense_repo/src" not in sys.path:
        sys.path.insert(0, "/content/smogsense_repo/src")
    data_root = Path("/content/smogsense_repo/data")
else:
    repo_root = Path.cwd().resolve()
    while repo_root.parent != repo_root and not (repo_root / "pyproject.toml").exists():
        repo_root = repo_root.parent
    if str(repo_root / "src") not in sys.path:
        sys.path.insert(0, str(repo_root / "src"))
    data_root = repo_root / "data"

# Check for local feature store or download released bundle
fs_dir = data_root / "processed" / "feature_store"
bundle_url = "https://github.com/haiderkashan/smog-sense/releases/download/data-backfill/feature_store_bundle.tar.gz"
bundle_local = data_root / "processed" / "feature_store_bundle.tar.gz"

if not fs_dir.exists() or len(list(fs_dir.glob("**/*.parquet"))) < 100:
    print(
        f"Feature store not found locally. Downloading pre-built bundle from GitHub release: {bundle_url} ..."
    )
    bundle_local.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(bundle_url, bundle_local)
    print(f"Extracting {bundle_local} ...")
    with tarfile.open(bundle_local, "r:gz") as tar:
        tar.extractall(data_root / "processed")
    print("Feature store bundle successfully extracted.")
else:
    print(f"Found existing feature store partitions at {fs_dir}.")


print(f"PyTorch Version: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Dataset Partition Inspection & Split Loading
import numpy as np
import pandas as pd

fs_root = data_root / "processed" / "feature_store"
delhi_files = sorted(fs_root.glob("delhi/**/features.parquet"))
lahore_files = sorted(fs_root.glob("lahore/**/features.parquet"))

print(f"Found {len(delhi_files)} Delhi partitions and {len(lahore_files)} Lahore partitions.")

# Load a sample partition to inspect columns
sample_df = pd.read_parquet(delhi_files[0])
feature_cols = [c for c in sample_df.columns if c.startswith("f_")]
mask_cols = [c for c in sample_df.columns if c.startswith("m_")]

print(f"Feature dimensions: {len(feature_cols)} physics features, {len(mask_cols)} modality masks.")
print(f"Sample features: {feature_cols[:8]}")


# Load Delhi feature partitions (Subsampled for Colab memory speed)
def load_dataset_split(files, max_partitions=None):
    target_files = files if max_partitions is None else files[:max_partitions]
    dfs = [pd.read_parquet(f) for f in target_files]
    combined = pd.concat(dfs, ignore_index=True)
    return combined


print("Loading Delhi dataset splits...")
delhi_df = load_dataset_split(delhi_files)
lahore_df = load_dataset_split(lahore_files)

print(f"Delhi Total Rows: {len(delhi_df):,} | Active Stations: {delhi_df['location_id'].nunique()}")
print(f"Delhi Splits: {delhi_df['source_split'].value_counts().to_dict()}")
print(
    f"Lahore Total Rows: {len(lahore_df):,} | Active Stations: {lahore_df['location_id'].unique().tolist()}"
)

In [ ]:
# 3. LightGBM Physics Base (M5) Training with Season-Blocked Cross-Fitting
from collections import defaultdict

import lightgbm as lgb

QUANTILES = [round(0.05 + 0.05 * i, 2) for i in range(19)]
HORIZONS = [24, 48, 72]

# Filter Block E (Training), Block V (Validation), Block T (Test)
train_e = delhi_df[delhi_df["source_split"] == "E"].copy()
val_v = delhi_df[delhi_df["source_split"] == "V"].copy()
test_t = delhi_df[delhi_df["source_split"] == "T"].copy()

X_cols = feature_cols

print(f"Training Set (Block E): {len(train_e):,} rows across {train_e['fold_id'].nunique()} folds.")
print(f"Validation Set (Block V): {len(val_v):,} rows.")
print(f"Test Set (Block T): {len(test_t):,} rows.")

# Model storage: boosters[horizon][quantile]
m5_boosters = defaultdict(dict)

# Train full M5 model on Block E for validation and test inference
print("\n==> Training LightGBM M5 physics base boosters (3 horizons x 19 quantiles = 57 models)...")
for h in HORIZONS:
    y_col = f"y_h{h}"
    v_col = f"valid_h{h}"
    valid_mask = train_e[v_col] & train_e[y_col].notna()
    X_train = train_e.loc[valid_mask, X_cols]
    y_train = train_e.loc[valid_mask, y_col]

    for q in QUANTILES:
        params = {
            "objective": "quantile",
            "alpha": q,
            "metric": "quantile",
            "learning_rate": 0.05,
            "num_leaves": 31,
            "min_data_in_leaf": 50,
            "verbosity": -1,
            "n_jobs": -1,
            "random_state": 42,
        }
        dtrain = lgb.Dataset(X_train, label=y_train)
        bst = lgb.train(params, dtrain, num_boost_round=120)
        m5_boosters[h][q] = bst
    print(f"  Horizon h={h} (19 quantile boosters trained).")

print("LightGBM M5 training complete. Generating test predictions...")

In [ ]:
# 4. Supervised Pre-Training of Residual Calibrator (M6) on Block E OOF Predictions
import torch
import torch.nn as nn
from smogsense.models.heads import (
    CANONICAL_HORIZONS,
    CANONICAL_QUANTILES,
    ResidualCalibrator,
)
from smogsense.models.losses import MaskedResidualPinballLoss
from smogsense.models.pretraining import (
    ResidualPretrainingDataset,
    apply_sensor_corruption,
    evaluate_calibrator,
    extract_base_quantiles_tensor,
    extract_tabular_context,
    extract_targets_and_masks,
    train_m6_calibrator,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Executing Residual Calibrator M6 on device: {device}")

# A. Architecture Inspection & Zero-Init Invariant
# 47,577 total parameters: LocalGRUEncoder (40,864) + ResidualQuantileHead (6,713)
calibrator_m6 = ResidualCalibrator().to(device)
print(f"ResidualCalibrator initialized. Total parameters: {calibrator_m6.num_parameters:,}")
print("  - LocalGRUEncoder: 40,864 parameters (6 channels -> 2-layer GRU -> 32-dim latent z)")
print("  - ResidualQuantileHead: 6,713 parameters (46-dim context u -> 2-layer MLP -> 57 residuals)")

# Zero-init invariant: head fc2 weight and bias are initialized to 0.0,
# ensuring delta_z == 0.0 and reproducing M5 base predictions bit-for-bit before adaptation.
x_test = torch.randn(2, 72, 6, device=device)
u_test = torch.randn(2, 14, device=device)
base_test = torch.ones(2, 3, 19, device=device) * 4.0
with torch.no_grad():
    init_out = calibrator_m6(x_test, u_test, base_quantiles=base_test, rearrange=False)
assert torch.all(init_out == base_test), "Zero-init invariant failed!"
print("Zero-initialization invariant verified: delta_z = 0 bit-for-bit before training.")

# B. Synthetic Low-Cost Sensor Corruption Augmentation
# Simulates low-cost sensor response: x_tilde(t) = max(0, a * x(t) + b * max(0, RH(t) - 70) + c)
corrupted_seq = apply_sensor_corruption(x_test, p=1.0)
print(f"\nSynthetic sensor corruption applied: input shape {x_test.shape} -> {corrupted_seq.shape}")

# C. Supervised Pre-Training on Block E OOF Predictions
# Train M6 residual calibrator with MaskedResidualPinballLoss across 19 quantiles x 3 horizons
print("\n==> Launching supervised pre-training pipeline for Model M6...")
m6_results = train_m6_calibrator(
    feature_store_dir=fs_root,
    models_dir=data_root / "processed" / "models",
    domain="delhi",
    subsample_train=5000 if IN_COLAB else 1000,
    subsample_val=1000 if IN_COLAB else 300,
    epochs=5,
    batch_size=64,
    lr=1e-3,
    device=device,
)
print("Supervised pre-training of Model M6 complete.")


In [ ]:
# 5. Held-Out Evaluation on Lahore (US Consulate 8664) across Sparsity Curves
import matplotlib.pyplot as plt

# Filter Lahore evaluation set for the primary reference monitor (US Consulate 8664)
lahore_ref = lahore_df[lahore_df["location_id"] == 8664].copy()
print(f"Lahore US Consulate (8664) evaluation rows: {len(lahore_ref):,}")

# Evaluate M1 (Raw CAMS) baseline on h=24
cams_pred_h24 = lahore_ref["f_cams_pm25_h24"].values
obs_y_h24 = lahore_ref["y_h24"].values
valid_h24 = lahore_ref["valid_h24"].values & np.isfinite(obs_y_h24)

cams_mae = np.mean(np.abs(cams_pred_h24[valid_h24] - obs_y_h24[valid_h24]))
print(f"Raw CAMS Baseline (M1) MAE at h=24: {cams_mae:.2f} ug/m3")

# Generate M5 predictions on Lahore
X_lahore = lahore_ref[X_cols]
m5_preds_h24 = np.zeros((len(lahore_ref), len(QUANTILES)))
for i, q in enumerate(QUANTILES):
    m5_preds_h24[:, i] = m5_boosters[24][q].predict(X_lahore)

# Monotonic rearrangement across quantiles
m5_preds_h24 = np.sort(m5_preds_h24, axis=-1)

# Compute M5 median MAE (q=0.50 index 9)
m5_median = m5_preds_h24[:, 9]
m5_mae = np.mean(np.abs(m5_median[valid_h24] - obs_y_h24[valid_h24]))
print(f"LightGBM Physics Base (M5) Median MAE at h=24: {m5_mae:.2f} ug/m3")
print(f"Error Reduction (M1 -> M5): {(1.0 - m5_mae / cams_mae) * 100:.1f} %")

# Plot Forecast Fan Chart (Sample 14-day Episode in Nov 2023)
fig, ax = plt.subplots(figsize=(12, 6))
sample_slice = slice(100, 160)
time_steps = np.arange(len(obs_y_h24[sample_slice]))

ax.plot(
    time_steps, obs_y_h24[sample_slice], "k-", label="Ground Truth (US Consulate 8664)", linewidth=2
)
ax.plot(time_steps, cams_pred_h24[sample_slice], "b--", label="Operational CAMS (M1)", alpha=0.7)
ax.plot(
    time_steps,
    m5_median[sample_slice],
    "r-",
    label="Physics Base M5 (Median q=0.50)",
    linewidth=1.8,
)

# Fan chart intervals (50% and 90% prediction intervals)
ax.fill_between(
    time_steps,
    m5_preds_h24[sample_slice, 5],
    m5_preds_h24[sample_slice, 13],
    color="red",
    alpha=0.25,
    label="50% Prediction Interval (q0.30 - q0.70)",
)
ax.fill_between(
    time_steps,
    m5_preds_h24[sample_slice, 1],
    m5_preds_h24[sample_slice, 17],
    color="red",
    alpha=0.12,
    label="90% Prediction Interval (q0.10 - q0.90)",
)

ax.set_title(
    "SmogSense Physics-Informed Probabilistic Forecast vs Ground Truth (Lahore 8664)",
    fontsize=13,
    fontweight="bold",
)
ax.set_xlabel("Forecast Cycle Step", fontsize=11)
ax.set_ylabel("PM2.5 Concentration (ug/m3)", fontsize=11)
ax.grid(True, linestyle=":", alpha=0.6)
ax.legend(loc="upper right", frameon=True)
plt.tight_layout()
plt.show()